# Debug Tableau Cloud Login
Run cell by cell to isolate where auth fails.

## Section 1: PAT Auth

In [ ]:
import os

server = "https://YOUR_POD.online.tableau.com"
api_version = "3.28"

pat_name = "your_pat_name"
pat_secret = os.environ.get("YOUR_PAT_SECRET_ENV_VAR")

print(f"PAT name:   {pat_name}")
print(f"PAT secret: {'SET (' + str(len(pat_secret)) + ' chars)' if pat_secret else 'NOT SET'}")

In [ ]:
from tableau_api_lib import TableauServerConnection

site = "your-site"

config = {
    "tableau_prod": {
        "server": server,
        "api_version": api_version,
        "site_name": site,
        "site_url": site,
        "personal_access_token_name": pat_name,
        "personal_access_token_secret": pat_secret,
    }
}

conn = TableauServerConnection(config_json=config, env="tableau_prod")
response = conn.sign_in()

print(f"Status: {response.status_code}")
print(f"Body:\n{response.text}")

In [ ]:
# If sign-in worked, try a simple API call
if response.status_code == 200:
    projects = conn.query_projects()
    print(f"query_projects status: {projects.status_code}")
    print(projects.json())
else:
    print("Skipped - sign-in failed above")

In [ ]:
try:
    conn.sign_out()
    print("Signed out")
except Exception as e:
    print(f"Sign-out: {e}")

---
## Section 2: Try a different site with PAT

In [ ]:
site2 = "your-other-site"

config2 = {
    "tableau_prod": {
        "server": server,
        "api_version": api_version,
        "site_name": site2,
        "site_url": site2,
        "personal_access_token_name": pat_name,
        "personal_access_token_secret": pat_secret,
    }
}

conn2 = TableauServerConnection(config_json=config2, env="tableau_prod")
resp2 = conn2.sign_in()

print(f"Site '{site2}' - Status: {resp2.status_code}")
print(resp2.text)

if resp2.status_code == 200:
    conn2.sign_out()

---
## Section 3: JWT Auth (Connected App)

Fill in your Connected App credentials from the Tableau Cloud UI.

To create: **Site Settings > Connected Apps > New > Direct Trust > Generate Secret**

In [ ]:
# Fill these in from your Connected App
jwt_client_id    = ""   # from the Connected App page
jwt_secret_id    = ""   # shown after generating a secret
jwt_secret_value = ""   # shown once when you generate — save it!
jwt_username     = ""   # email of the admin user to impersonate
jwt_site         = "your-site"  # content_url of the site

print(f"Client ID:    {'SET' if jwt_client_id else 'EMPTY'}")
print(f"Secret ID:    {'SET' if jwt_secret_id else 'EMPTY'}")
print(f"Secret Value: {'SET' if jwt_secret_value else 'EMPTY'}")
print(f"Username:     {jwt_username or 'EMPTY'}")
print(f"Site:         {jwt_site}")

In [ ]:
import jwt
import time
import uuid

# Step 1: Mint the JWT
now = int(time.time())
payload = {
    "iss": jwt_client_id,
    "sub": jwt_username,
    "aud": "tableau",
    "exp": now + 300,
    "jti": str(uuid.uuid4()),
    "scp": ["tableau:content:read",
            "tableau:tasks:run",
            "tableau:workbooks:create",
            "tableau:datasources:create",
            "tableau:projects:create",
            "tableau:groups:create",
            "tableau:users:create"],
}
token = jwt.encode(payload, jwt_secret_value, algorithm="HS256", headers={
    "kid": jwt_secret_id,
    "iss": jwt_client_id,
})

print(f"JWT minted ({len(token)} chars)")
print(f"First 50 chars: {token[:50]}...")

In [ ]:
import requests

# Step 2: Sign in with the JWT
url = f"{server}/api/{api_version}/auth/signin"
body = {
    "credentials": {
        "jwt": token,
        "site": {
            "contentUrl": jwt_site,
        },
    }
}

resp = requests.post(url, json=body)
print(f"Status: {resp.status_code}")
print(f"Body:\n{resp.text}")

In [ ]:
# Step 3: If JWT sign-in worked, use the auth token with tableau_api_lib
if resp.status_code == 200:
    creds = resp.json()["credentials"]
    auth_token = creds["token"]
    site_id = creds["site"]["id"]
    user_id = creds["user"]["id"]
    print(f"Auth token: {auth_token[:20]}...")
    print(f"Site ID:    {site_id}")
    print(f"User ID:    {user_id}")

    # Inject into a TableauServerConnection
    jwt_conn = TableauServerConnection(
        config_json={
            "tableau_prod": {
                "server": server,
                "api_version": api_version,
                "site_name": jwt_site,
                "site_url": jwt_site,
                "personal_access_token_name": "jwt_placeholder",
                "personal_access_token_secret": "jwt_placeholder",
            }
        },
        env="tableau_prod",
    )
    jwt_conn.auth_token = auth_token
    jwt_conn.site_id = site_id
    jwt_conn.user_id = user_id

    # Test: query projects on this site
    projects = jwt_conn.query_projects()
    print(f"\nquery_projects status: {projects.status_code}")
    print(projects.json())

    jwt_conn.sign_out()
    print("\nSigned out")
else:
    print("Skipped - JWT sign-in failed above")